<a href="https://colab.research.google.com/github/eduzegarra/grade_01/blob/main/info_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [144]:
import pandas as pd
info=pd.read_stata('/content/drive/MyDrive/a_SENAMHI/OUT/informacion.dta')
caratulas=pd.read_stata('/content/drive/MyDrive/a_SENAMHI/OUT/caratulas.dta')

In [ ]:
caratulas.cod_prod

In [145]:
info_clima=((info.filter(regex=r'^info_5|_05$|AGROCLIM?|cod_prod')))

In [146]:
merge_01=caratulas.merge(info_clima, on='cod_prod', how='left')

In [ ]:
merge_01

In [147]:
pp=merge_01.groupby(['LATITUD','LONGITUD'], as_index=False).agg({
                                         'FACTOR_PRODUCTOR':'count'}).rename(columns={'FACTOR_PRODUCTOR':'num_prod'})

In [148]:
len(pp), len(merge_01)

(25403, 112866)

In [ ]:
info_clima.filter(regex=r'^info_5|_05$|AGROCLIM?').isna().sum()

In [ ]:
merge_01.filter(regex=r'^info_5|_05$|AGROCLIM?').columns

In [149]:
group_01=merge_01.groupby(['LATITUD','LONGITUD'], as_index=False).agg({'info_5':'mean',
       'inst_MIDAGRI_05':'mean', 'inst_AMIGO_05':'mean', 'inst_SENAMHI_05':'mean',
       'inst_EMPRESA_05':'mean', 'inst_GORE_05':'mean', 'inst_MUNI_05':'mean', 'inst_ASOC_05':'mean',
       'medio_RADIO_05':'mean', 'medio_TV_05':'mean', 'medio_INTERNET_05':'mean', 'medio_TELEF_05':'mean',
       'medio_VERBAL_05':'mean', 'neces_AGROCLIM':'mean'})

In [150]:
len(group_01)

25403

In [151]:
merge_02=group_01.merge(pp, on=['LATITUD','LONGITUD'], how='left')

In [ ]:
merge_02.columns

In [152]:
import geopandas
from shapely.geometry import Point

# Create a geometry column from LATITUD and LONGITUD
merge_02['geometry'] = merge_02.apply(lambda row: Point(row['LONGITUD'],
          row['LATITUD']) if pd.notnull(row['LONGITUD']) and pd.notnull(row['LATITUD']) else None, axis=1)